# Figure 2: Cell and gene clustering

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


## Fig 2 | Clustering & dendrogram

In [ ]:
import anndata as ad
import numpy as np
adata = ad.read_h5ad("../../data/pbmc_cite_seq.h5ad")

# Match the exported IDs: training can remove unexpressed genes.
adata = adata[
    np.load("../../results/pbmc_cite_seq/scLDM_32D_laplacian/scLDM_cell_names.npy"),
    np.load("../../results/pbmc_cite_seq/scLDM_32D_laplacian/scLDM_gene_names.npy"),
].copy()

z_cells = np.load('../../results/pbmc_cite_seq/scLDM_32D_laplacian/scLDM_cell_latent.npy')
z_genes = np.load('../../results/pbmc_cite_seq/scLDM_32D_laplacian/scLDM_gene_latent.npy')

if z_cells.shape[1] != z_genes.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells.shape}, genes={z_genes.shape}")

adata.obsm['scLDM'] = z_cells
adata.varm['scLDM'] = z_genes


z_cells_2d = np.load('../../results/pbmc_cite_seq/scLDM_2D/scLDM_cell_latent.npy')
z_genes_2d = np.load('../../results/pbmc_cite_seq/scLDM_2D/scLDM_gene_latent.npy')

if z_cells_2d.shape[1] != z_genes_2d.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_2d.shape}, genes={z_genes_2d.shape}")

adata.obsm['scLDM_2D'] = z_cells_2d
adata.varm['scLDM_2D'] = z_genes_2d

In [ ]:
from scipy import sparse
from scipy.sparse.linalg import svds
import numpy as np

def robust_scale_embedding(U, target=0.1, q=0.99, clip=0.5, eps=1e-8):
    """
    Robust column-wise scaling for spectral initialization.

    After scaling, the q-th quantile of |U| is approximately `target`.
    Values are clipped to avoid extreme localized eigenvector entries.
    """
    U = U - np.median(U, axis=0, keepdims=True)

    scale = np.quantile(np.abs(U), q, axis=0, keepdims=True)
    U = target * U / (scale + eps)

    if clip is not None:
        U = np.clip(U, -clip, clip)

    return U

def bipartite_laplacian_from_svd(B: sparse.csr_matrix, k: int, tol: float = 1e-4, rng: int = 42):
    m, n = B.shape
    du = np.asarray(B.sum(axis=1)).ravel()
    dv = np.asarray(B.sum(axis=0)).ravel()

    # protect against zeros
    du_safe = np.where(du == 0, 1.0, du)
    dv_safe = np.where(dv == 0, 1.0, dv)

    Du_mhalf = sparse.diags(1.0 / np.sqrt(du_safe))
    Dv_mhalf = sparse.diags(1.0 / np.sqrt(dv_safe))

    B_tilde = Du_mhalf @ B @ Dv_mhalf          # (m × n)

    # 1) Ask for k+1 largest‐magnitude singular values,
    #    so we pick up the trivial σ≈1 mode as well.
    u_all, s_all, vt_all = svds(B_tilde, k=k+1, which="LM", tol=tol, random_state=np.random.default_rng(rng))

    # 2) Sort descending by σ:
    idx_all = np.argsort(s_all)[::-1]
    s_all = s_all[idx_all]
    u_all = u_all[:, idx_all]
    vt_all = vt_all[idx_all, :]

    # 3) Drop the very first singular value/vector (the trivial σ≈1).
    #    Keep the next k largest.
    s = s_all[1:]       # length k
    u = u_all[:, 1:]    # (m × k)
    vt = vt_all[1:, :]  # (k × n)

    lam = 1.0 - s                               # λ = 1 − σ

    # Cell and gene coordinate blocks
    # left  : (m × k)   right : (n × k)
    left  = (u * (1.0 / np.sqrt(du_safe))[:, None]) / np.sqrt(2.0)
    right = (vt.T * (1.0 / np.sqrt(dv_safe))[:, None]) / np.sqrt(2.0)
    # ---------------------------------------------------------------------

    # symmetric normalized spectral embedding
    U = np.concatenate([left, right], axis=0)

    # robust scaling instead of ordinary z-scoring
    #U = robust_scale_embedding(U, target=0.1, q=0.99, clip=0.5) # Optional embedding rescaling

    return lam, U



def laplacian_init(B: sparse.csr_matrix, k: int, seed: int = 42):
    """
    Convenience wrapper that splits the eigenvectors into
    separate cell- and gene-blocks ready to pass to scLDM.
    """
    _, U = bipartite_laplacian_from_svd(B, k, rng=seed)
    n_cells = B.shape[0]
    Z_cells = U[:n_cells, :].astype(np.float32)
    Z_genes = U[n_cells:, :].astype(np.float32)
    return Z_cells, Z_genes
import anndata as ad

# Load the count matrix
z_init_cells, z_init_genes = laplacian_init(adata.X, k=32, seed=42)
adata.obsm["z_init_cells"] = z_init_cells
adata.varm["z_init_genes"] = z_init_genes


# Load the count matrix
z_init_cells_2d, z_init_genes_2d = laplacian_init(adata.X, k=2, seed=42)
adata.obsm["z_init_cells_2D"] = z_init_cells_2d
adata.varm["z_init_genes_2D"] = z_init_genes_2d

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

# ----- 1.  Build a placeholder expression matrix for the genes -----
# Gene rows have zero-filled expression values; their embeddings are stored separately.
n_cells, n_genes = adata.n_obs, adata.n_vars
if sp.issparse(adata.X):
    placeholder_X = sp.csr_matrix((n_genes, n_genes), dtype=adata.X.dtype)
else:
    placeholder_X = np.zeros((n_genes, n_genes), dtype=adata.X.dtype)

# ----- 2.  Stack the real cells with the pseudo-cells (genes) -----
X_combined = (
    sp.vstack([adata.X, placeholder_X])
    if sp.issparse(adata.X)
    else np.vstack([adata.X, placeholder_X])
)

# ----- 3.  Create an .obs that labels each row as cell / gene -----
obs_combined = pd.concat(
    [
        adata.obs.assign(entity="cell"),               # keep existing cell metadata
        pd.DataFrame({"entity": "gene"}, index=adata.var_names)  # one row per gene
    ]
)

# ----- 4.  Assemble the new AnnData object -----
adata_combo = sc.AnnData(
    X=X_combined,
    obs=obs_combined,
    var=adata.var.copy()            # keep original gene metadata as .var
)

# ----- 5.  Concatenate the embeddings and store in .obsm -----
adata_combo.obsm["z_init_cells"] = np.vstack([
    adata.obsm["z_init_cells"],      # cells (n_cells × dim)
    adata.varm["z_init_genes"]       # genes (n_genes × dim)
])

adata_combo.obsm["scLDM"] = np.vstack([
    adata.obsm["scLDM"],      # cells (n_cells × dim)
    adata.varm["scLDM"]       # genes (n_genes × dim)
])


adata_combo.obsm["z_init_cells_2D"] = np.vstack([
    adata.obsm["z_init_cells_2D"],      # cells (n_cells × dim)
    adata.varm["z_init_genes_2D"]       # genes (n_genes × dim)
])

adata_combo.obsm["scLDM_2D"] = np.vstack([
    adata.obsm["scLDM_2D"],      # cells (n_cells × dim)
    adata.varm["scLDM_2D"]       # genes (n_genes × dim)
])


In [ ]:
import scanpy as sc
sc.pp.neighbors(adata_combo, use_rep="z_init_cells")
sc.tl.umap(adata_combo, random_state=42)

In [ ]:
sc.pl.umap(adata_combo, color=["celltype.l1"])

In [ ]:
import numpy as np
import pandas as pd
import fastcluster
from scipy.cluster.hierarchy import fcluster, leaves_list

X = np.asarray(adata_combo.obsm["scLDM"], dtype=np.float64)

Z = fastcluster.linkage_vector(
    X,
    method="ward",
    metric="euclidean",
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

K = int(np.ceil(np.log2(len(adata_combo))))
labels = fcluster(Z, t=K, criterion="maxclust")

cat_order = [str(i) for i in range(1, K + 1)]
adata_combo.obs["ward_scLDM"] = pd.Categorical(
    labels.astype(str),
    categories=cat_order,
    ordered=True
)

# one palette used everywhere
cmap = plt.get_cmap("tab20", K)
palette_str = {str(i): to_hex(cmap(i - 1)) for i in range(1, K + 1)}  # for obs (strings)
palette_int = {i: palette_str[str(i)] for i in range(1, K + 1)}        # for fcluster ints


In [ ]:
def plot_dendrogram(
    linkage_matrix,
    plots_folder,
    max_levels=None,
    filename="dendrogram",
    n_clusters=None,
    distance_threshold=None,
    colors=None,
):
    import os
    import numpy as np
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from math import ceil, log2, floor
    from scipy.cluster.hierarchy import dendrogram, fcluster
    from matplotlib.ticker import FuncFormatter

    # Plot typography and export settings
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    # Final-size typography targets
    LABEL_FS = 7.0      # allowed max for normal text
    TICK_FS = 5.5       # allowed min is 5 pt
    BRANCH_LW = 0.3
    SPINE_LW = 0.5
    TICK_LW = 0.5
    TICK_LEN = 2.5
    TICK_LEN = 3

    # Number of original observations = number of leaves
    n = linkage_matrix.shape[0] + 1

    dendro_data = dendrogram(
        linkage_matrix,
        leaf_rotation=90,
        truncate_mode="level" if max_levels is not None else None,
        p=max_levels if max_levels is not None else 0,
        no_plot=True,
        distance_sort="ascending",
    )

    cluster_labels = fcluster(linkage_matrix, t=n_clusters, criterion="maxclust")
    leaf_clusters = cluster_labels - 1  # Square panel dimensions

    if colors is None:
        unique_clusters = np.unique(cluster_labels)
        cmap = plt.get_cmap("tab20", len(unique_clusters))
        cluster_colors = {cluster: cmap(i) for i, cluster in enumerate(unique_clusters)}
    else:
        cluster_colors = colors

    leaves_order = dendro_data["leaves"]
    x_positions = [5 + 10 * i for i in range(len(leaves_order))]
    x_to_leaf = dict(zip(x_positions, leaves_order))

    branch_colors = []
    for i in range(len(dendro_data["icoord"])):
        icoord = dendro_data["icoord"][i]
        dcoord = dendro_data["dcoord"][i]

        x1, x2 = icoord[1], icoord[2]
        leaf1_pos = int(round((x1 - 5.0) / 10.0))
        leaf2_pos = int(round((x2 - 5.0) / 10.0))

        leaf1_idx = dendro_data["leaves"][leaf1_pos] if 0 <= leaf1_pos < len(leaves_order) else None
        leaf2_idx = dendro_data["leaves"][leaf2_pos] if 0 <= leaf2_pos < len(leaves_order) else None

        if leaf1_idx is not None and leaf2_idx is not None:
            if cluster_labels[leaf1_idx] == cluster_labels[leaf2_idx]:
                color = cluster_colors[cluster_labels[leaf1_idx]]
            else:
                color = "k"
        else:
            color = "k"

        branch_colors.append(color)

    # Use a panel size closer to final output size
    fig, ax = plt.subplots(figsize=(6.5, 1.4))

    stop_level = distance_threshold

    for i in range(len(dendro_data["dcoord"])):
        dendro_data["dcoord"][i] = np.square(dendro_data["dcoord"][i])

    for xs, ys, color in zip(dendro_data["icoord"], dendro_data["dcoord"], branch_colors):
        if stop_level is None or max(ys) >= stop_level:
            ax.plot(xs, ys, color=color, linewidth=BRANCH_LW, solid_capstyle="butt")

    if stop_level is not None:
        leaf_width = 10
        old_x = np.unique(np.concatenate([line.get_xdata() for line in ax.get_lines()]))
        old_x.sort()

        new_x = leaf_width / 2 + leaf_width * np.arange(len(old_x))

        from scipy.interpolate import interp1d
        xmap = interp1d(old_x, new_x, kind="linear", fill_value="extrapolate")

        for line in ax.get_lines():
            line.set_xdata(xmap(line.get_xdata()))

        ax.set_xlim(0, leaf_width * len(old_x))

    # Change this label only if units exist
    ax.set_ylabel(r"$\log_2$-SED", fontsize=LABEL_FS)

    ax.set_yscale("log", base=2)

    y_min, y_max = ax.get_ylim()
    if y_min <= 0:
        y_min = 1

    min_exp = floor(log2(y_min))
    max_exp = ceil(log2(y_max))
    y_ticks = [2**i for i in range(min_exp, max_exp + 1)]

    ax.set_yticks(y_ticks)
    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda y, _: f"{int(np.log2(y))}" if y > 0 else "")
    )

    ax.tick_params(
        axis="y",
        labelsize=TICK_FS,
        width=TICK_LW,
        length=TICK_LEN,
        pad=2,
    )

    ax.set_xticks([])
    ax.set_xlabel("")

    # Keep the left axis visible; remove only top/right
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_linewidth(SPINE_LW)
    ax.spines["left"].set_position(("outward", 4))
    plt.tight_layout(pad=0.2)

    plot_path = os.path.join(plots_folder, filename)

    # SVG export
    plt.savefig(plot_path + ".svg", bbox_inches="tight", format="svg", dpi=600)

    # Optional preview export
    #plt.savefig(plot_path + ".png", bbox_inches="tight", format="png", dpi=450)

    plt.show()
    print(f"Dendrogram plot saved at: {plot_path}")

In [ ]:
plot_dendrogram(
    linkage_matrix=Z,
    plots_folder='output/fig_2',
    filename='pbmc_cite_seq_dendrogram',
    n_clusters=K,
    distance_threshold = 120,
    colors = palette_int
    #max_levels = max_depth_cells
)


In [ ]:
import scanpy as sc
sc.pp.neighbors(adata_combo, use_rep="z_init_cells")
sc.tl.umap(adata_combo, random_state=42, key_added="umap_init")

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata_combo, use_rep="scLDM")
sc.tl.umap(adata_combo, random_state=42, key_added="umap_scLDM")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import numpy as np
import matplotlib as mpl


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# Extract coordinates
umap = adata_combo.obsm["umap_init"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

celltypes = adata_combo.obs["ward_scLDM"]
categories = list(celltypes.cat.categories)

fig, ax = plt.subplots(figsize=(2.36, 2.00))

# Plot cells by cell type
for ct in categories:
    idx = (celltypes == ct).values
    ax.scatter(
        umap[idx, 0],
        umap[idx, 1],
        s=0.1,
        c=[palette_str[ct]],
        alpha=1,
        marker='.',
        label=ct,
        linewidths=0,
        rasterized=True
    )

# Optional: label genes
# gene_names = adata_combo.obs_names[gene_mask]
# for (x, y), name in zip(genes, gene_names):
#     ax.text(x, y, name, fontsize=6, color="black", alpha=0.9)

ax.set_axis_off()
ax.set_title("")
#ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=1.5)

plt.savefig("output/fig_2/pbmc_cite_seq_umap_laplacian_ward.svg", bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()
#plt.close()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import numpy as np

# Extract coordinates
umap = adata_combo.obsm["umap_scLDM"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

celltypes = adata_combo.obs["ward_scLDM"]
categories = list(celltypes.cat.categories)

fig, ax = plt.subplots(figsize=(2.36, 2.00))

# Plot cells by cell type
for ct in categories:
    idx = (celltypes == ct).values
    ax.scatter(
        umap[idx, 0],
        umap[idx, 1],
        s=0.1,
        c=[palette_str[ct]],
        alpha=1,
        marker='.',
        label=ct,
        linewidths=0,
        rasterized=True
    )

# Optional: label genes
# gene_names = adata_combo.obs_names[gene_mask]
# for (x, y), name in zip(genes, gene_names):
#     ax.text(x, y, name, fontsize=6, color="black", alpha=0.9)

ax.set_axis_off()
ax.set_title("")
#ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=1.5)

plt.savefig("output/fig_2/pbmc_cite_seq_umap_latent_ward.svg", bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()
#plt.close()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal

# Extract coordinates
umap = adata_combo.obsm["umap_init"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

celltypes = adata_combo.obs.loc[cell_mask, "celltype.l1"]

# Build a palette for cell types
categories = pd.Categorical(celltypes).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette = dict(zip(categories, colors))

fig, ax = plt.subplots(figsize=(2.36, 2.00))

# Plot cells by cell type
for ct in categories:
    idx = (celltypes == ct).values
    ax.scatter(
        cells[idx, 0],
        cells[idx, 1],
        s=0.1,
        c=[palette[ct]],
        alpha=1,
        marker='.',
        label=ct,
        linewidths=0,
        rasterized=True
    )

# Overlay genes in one explicit color
ax.scatter(
    genes[:, 0],
    genes[:, 1],
    s=0.1,
    c="#7A1E1E",
    alpha=1,
    marker='.',
    label="genes",
    linewidths=0,
    rasterized=True
)

# Optional: label genes
# gene_names = adata_combo.obs_names[gene_mask]
# for (x, y), name in zip(genes, gene_names):
#     ax.text(x, y, name, fontsize=6, color="black", alpha=0.9)

ax.set_axis_off()
ax.set_title("")
#ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=1.5)

plt.savefig("output/fig_2/pbmc_cite_seq_umap_laplacian_celltype.svg", bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()
#plt.close()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal

# Extract coordinates
umap = adata_combo.obsm["umap_scLDM"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

celltypes = adata_combo.obs.loc[cell_mask, "celltype.l1"]

# Build a palette for cell types
categories = pd.Categorical(celltypes).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette = dict(zip(categories, colors))

fig, ax = plt.subplots(figsize=(2.36, 2.00))

# Plot cells by cell type
for ct in categories:
    idx = (celltypes == ct).values
    ax.scatter(
        cells[idx, 0],
        cells[idx, 1],
        s=0.1,
        c=[palette[ct]],
        alpha=1,
        marker='.',
        label=ct,
        linewidths=0,
        rasterized=True
    )

# Overlay genes in one explicit color
ax.scatter(
    genes[:, 0],
    genes[:, 1],
    s=0.1,
    c="#7A1E1E",
    alpha=1,
    marker='.',
    label="genes",
    linewidths=0,
    rasterized=True
)

# Optional: label genes
# gene_names = adata_combo.obs_names[gene_mask]
# for (x, y), name in zip(genes, gene_names):
#     ax.text(x, y, name, fontsize=6, color="black", alpha=0.9)

ax.set_axis_off()
ax.set_title("")
#ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=1.5)

plt.savefig("output/fig_2/pbmc_cite_seq_umap_latent_celltype.svg", bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()
#plt.close()

In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def save_cluster_legend_pdf(
    categories,
    palette,
    output_pdf,
    ncol=1,
    fontsize=6,
    marker_size=4.0,
    columnspacing=0.8,
    handletextpad=0.4,
    labelspacing=0.35,
    borderpad=0.2,
):
    """
    Save a standalone legend-only PDF for cluster colors.

    Parameters
    ----------
    categories : list[str]
        Ordered category names.
    palette : dict
        Mapping {category: color}.
    output_pdf : str
        Output path ending in .pdf.
    ncol : int
        Number of legend columns.
    fontsize : float
        Legend text size in pt. Nature-style target: ~5–7 pt.
    marker_size : float
        Marker size in pt for legend keys.
    """

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    handles = [
        Line2D(
            [0], [0],
            linestyle="None",
            marker="o",
            markersize=marker_size,
            markerfacecolor=palette[cat],
            markeredgecolor=palette[cat],
            markeredgewidth=0.0,
            label=str(cat),
        )
        for cat in categories
    ]

    # Rough figure size estimate so the legend lays out predictably.
    n_items = len(categories)
    n_rows = math.ceil(n_items / ncol)
    fig_w = max(1.2, 1.15 * ncol + 0.55 * ncol)
    fig_h = max(0.35, 0.22 * n_rows + 0.18)

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.legend(
        handles=handles,
        labels=categories,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=0.8,
        handletextpad=handletextpad,
        columnspacing=columnspacing,
        labelspacing=labelspacing,
        borderpad=borderpad,
        markerscale=1.0,
    )

    fig.savefig(
        output_pdf,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )
    plt.close(fig)

In [ ]:
categories = list(adata_combo.obs["ward_scLDM"].cat.categories)

save_cluster_legend_pdf(
    categories=categories,
    palette=palette_str,
    output_pdf="output/fig_2/pbmc_cite_seq_umap_cluster_legend.svg",
    ncol=2,          # Number of legend columns
    fontsize=6,
    marker_size=4.0,
)

In [ ]:
categories = list(adata_combo.obs["celltype.l1"].cat.categories) + ["Gene"]

palette_legend = palette.copy()
palette_legend["Gene"] = "#7A1E1E"

save_cluster_legend_pdf(
    categories=categories,
    palette=palette_legend,
    output_pdf="output/fig_2/pbmc_cite_seq_umap_celltype_legend.svg",
    ncol=1,
    fontsize=6,
    marker_size=4.0,
)

## Cluster x Celltype heatmap

In [ ]:
from scipy.cluster.hierarchy import leaves_list
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Build a joint annotation:
# - cells keep their reference cell type
# - genes get the label "Gene"
# ---------------------------------------------------------
joint_annot = adata_combo.obs[["ward_scLDM", "entity"]].copy()

joint_annot["celltype_or_gene"] = np.where(
    joint_annot["entity"].eq("gene"),
    "Gene",
    adata_combo.obs["celltype.l1"].astype(object)
)

joint_annot = joint_annot.dropna(subset=["ward_scLDM", "celltype_or_gene"])

# ---------------------------------------------------------
# Order Ward clusters by dendrogram leaf order
# ---------------------------------------------------------
leaf_order = leaves_list(Z)
ward_order = pd.unique(pd.Series(labels[leaf_order]).astype(str)).tolist()

# Keep only clusters actually present
present_ward = joint_annot["ward_scLDM"].astype(str).unique().tolist()
ward_order = [w for w in ward_order if w in present_ward]

joint_annot["ward_scLDM"] = pd.Categorical(
    joint_annot["ward_scLDM"].astype(str),
    categories=ward_order,
    ordered=True,
)

# ---------------------------------------------------------
# Raw counts table: Ward cluster x (cell type OR Gene)
# ---------------------------------------------------------
ward_x_celltype_counts = pd.crosstab(
    joint_annot["ward_scLDM"],
    joint_annot["celltype_or_gene"],
    dropna=False,
).reindex(index=ward_order, fill_value=0)

# ---------------------------------------------------------
# Column order:
# biological labels first, Gene last
# ---------------------------------------------------------
non_gene_cols = [c for c in ward_x_celltype_counts.columns if c != "Gene"]

peak_cluster = ward_x_celltype_counts[non_gene_cols].idxmax(axis=0)
peak_rank = peak_cluster.map({w: i for i, w in enumerate(ward_order)})

celltype_order = (
    pd.DataFrame({
        "celltype": non_gene_cols,
        "peak_rank": peak_rank.values,
        "abundance": ward_x_celltype_counts[non_gene_cols].sum(axis=0).values,
    })
    .sort_values(["peak_rank", "abundance"], ascending=[True, False])
    ["celltype"]
    .tolist()
)

# Put Gene at the end
if "Gene" in ward_x_celltype_counts.columns:
    celltype_order = celltype_order + ["Gene"]

ward_x_celltype_counts = ward_x_celltype_counts.loc[:, celltype_order]

# ---------------------------------------------------------
# Row-normalized fractions:
# Fraction of each Ward cluster assigned to each label, including Gene.
# ---------------------------------------------------------
ward_x_celltype_frac = ward_x_celltype_counts.div(
    ward_x_celltype_counts.sum(axis=1).replace(0, np.nan),
    axis=0,
).fillna(0)

# ---------------------------------------------------------
# Cluster composition summary
# ---------------------------------------------------------
ward_summary = pd.DataFrame({
    "n_total": ward_x_celltype_counts.sum(axis=1),
    "n_genes": ward_x_celltype_counts["Gene"] if "Gene" in ward_x_celltype_counts.columns else 0,
    "gene_fraction": ward_x_celltype_frac["Gene"] if "Gene" in ward_x_celltype_frac.columns else 0.0,
    "dominant_label": ward_x_celltype_frac.idxmax(axis=1),
    "dominant_fraction": ward_x_celltype_frac.max(axis=1),
})

display(ward_summary.sort_index())

In [ ]:
def plot_ward_celltype_heatmap(
    frac_df,
    counts_df,
    output_pdf,
    ward_palette=None,
    celltype_palette=None,
    cmap="Blues",
    fig_w=3.2,
    fig_h=2.6,
):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    n_rows, n_cols = frac_df.shape

    # hardcoded figure size
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        frac_df.values,
        aspect="auto",
        cmap=cmap,
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(frac_df.columns, rotation=45, fontsize=5)

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(
        [f"{cl}" for cl in frac_df.index],
        fontsize=5,
    )

    #ax.set_xlabel("Reference entity", fontsize=6)
    ax.set_ylabel("Ward cluster", fontsize=6)

    max_j = frac_df.values.argmax(axis=1)
    ax.scatter(
        max_j,
        np.arange(n_rows),
        facecolors="white",
        edgecolors="white",
        linewidths=0.45,
        s=5,
        zorder=3,
    )

    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.tick_params(axis="x", length=0, pad=3)
    ax.tick_params(axis="y", length=0, pad=3)

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    if ward_palette is not None:
        for tick in ax.get_yticklabels():
            cluster_id = tick.get_text().split()[0]
            if cluster_id in ward_palette:
                tick.set_color(ward_palette[cluster_id])

    if celltype_palette is not None:
        for tick in ax.get_xticklabels():
            ct = tick.get_text()
            if ct in celltype_palette:
                tick.set_color(celltype_palette[ct])

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Fraction within Ward cluster", fontsize=6)
    cbar.ax.tick_params(labelsize=5, width=0.5, length=2)
    cbar.outline.set_linewidth(0.5)

    plt.tight_layout(pad=0.3)
    fig.savefig(output_pdf, transparent=True, dpi=450)
    plt.show()
    plt.close(fig)

In [ ]:
import os
os.makedirs("output/fig_2", exist_ok=True)

celltype_palette = palette.copy() if "palette" in globals() else {}
celltype_palette["Gene"] = "#7A1E1E"

plot_ward_celltype_heatmap(
    frac_df=ward_x_celltype_frac,
    counts_df=ward_x_celltype_counts,
    output_pdf="output/fig_2/pbmc_cite_seq_ward_x_celltype_heatmap_with_gene.svg",
    ward_palette=palette_str,
    celltype_palette=celltype_palette,
    cmap="Blues",
    fig_w=2.0,
    fig_h=3.2,

)

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

counts_no_genes = ward_x_celltype_counts.drop(columns=["Gene"], errors="ignore")

ward_labels = []
celltype_labels = []

for ward_cluster, row in counts_no_genes.iterrows():
    for celltype, count in row.items():
        ward_labels.extend([ward_cluster] * int(count))
        celltype_labels.extend([celltype] * int(count))

nmi = normalized_mutual_info_score(ward_labels, celltype_labels)

print(f"NMI excluding genes: {nmi:.4f}")


## 2D

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal

# Extract coordinates
umap = adata_combo.obsm["scLDM_2D"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

celltypes = adata_combo.obs.loc[cell_mask, "celltype.l1"]

# Build a palette for cell types
categories = pd.Categorical(celltypes).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette = dict(zip(categories, colors))

fig, ax = plt.subplots(figsize=(2.36, 2.00))

# Plot cells by cell type
for ct in categories:
    idx = (celltypes == ct).values
    ax.scatter(
        cells[idx, 0],
        cells[idx, 1],
        s=0.1,
        c=[palette[ct]],
        alpha=1,
        marker='.',
        label=ct,
        linewidths=0
    )

# Overlay genes in one explicit color
ax.scatter(
    genes[:, 0],
    genes[:, 1],
    s=0.1,
    c="#7A1E1E",
    alpha=1,
    marker='.',
    label="genes",
    linewidths=0
)

# Optional: label genes
# gene_names = adata_combo.obs_names[gene_mask]
# for (x, y), name in zip(genes, gene_names):
#     ax.text(x, y, name, fontsize=6, color="black", alpha=0.9)

ax.set_axis_off()
ax.set_title("")
#ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=1.5)

plt.savefig("output/fig_2/pbmc_cite_seq_umap_latent_celltype_2D.png", bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()
#plt.close()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal

# Extract coordinates
umap = adata_combo.obsm["z_init_cells_2D"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

celltypes = adata_combo.obs.loc[cell_mask, "celltype.l1"]

# Build a palette for cell types
categories = pd.Categorical(celltypes).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette = dict(zip(categories, colors))

fig, ax = plt.subplots(figsize=(5, 5.00))

# Plot cells by cell type
for ct in categories:
    idx = (celltypes == ct).values
    ax.scatter(
        cells[idx, 0],
        cells[idx, 1],
        s=0.3,
        c=[palette[ct]],
        alpha=1,
        marker='.',
        label=ct,
        linewidths=0
    )

# Overlay genes in one explicit color
ax.scatter(
    genes[:, 0],
    genes[:, 1],
    s=0.1,
    c="#7A1E1E",
    alpha=1,
    marker='.',
    label="genes",
    linewidths=0
)

# Optional: label genes
# gene_names = adata_combo.obs_names[gene_mask]
# for (x, y), name in zip(genes, gene_names):
#     ax.text(x, y, name, fontsize=6, color="black", alpha=0.9)

#ax.set_axis_off()
ax.set_title("")
ax.set_ylim(-.1, .1)
ax.set_xlim(-.1, .1)

#ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=1.5)

plt.savefig("output/fig_2/pbmc_cite_seq_umap_latent_celltype_2D.png", bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()
#plt.close()